# 📘 Notebook 1 — 基礎多媒體：相機與音訊

> 🎯 **目標**：學習擷取影像、錄製音訊和播放聲音 — 讓 Reachy 看得見、聽得到！

---

## 0. 你將學到什麼

在本教學結束時，你將能夠：

* 📸 從 Reachy 的相機擷取影像
* 🎬 顯示即時視訊畫面
* 🎤 從麥克風陣列錄製音訊
* 🔊 透過揚聲器播放聲音
* 💾 儲存與載入多媒體檔案
* 🤖 透過視聽反饋讓 Reachy 感覺更加「生動」

**預計時間：** 20 分鐘

> **開始之前：**  
> 請確保你在進入本教學前，已經完成 [`README.md`](README.md) 中環境需求區塊的所有設定步驟！

**注意：** 本教學需要相機與麥克風功能正常運作。這次請確保**不要**使用 `media_backend="no_media"`！

---

## 1. 環境設定

首先，讓我們匯入所需的函式庫。

In [ ]:
import time

# 相機影像處理函式庫
try:
    import cv2
except ImportError as e:
    print(f"⚠ Missing library: {e}")
    print("請在虛擬環境中安裝：uv pip install \"reachy-mini[opencv]\"")

import numpy as np
from IPython.display import Image as IPImage
from IPython.display import display

from reachy_mini import ReachyMini
from reachy_mini.utils import create_head_pose

# 音訊處理函式庫
try:
    import soundfile as sf

    print("✓ 所有函式庫匯入成功！")
except ImportError as e:
    print(f"⚠ Missing library: {e}")
    print("Install with: pip install soundfile scipy")

---

## 第 1 部分：相機基礎 📸

### 2. 擷取你的第一張影像

讓我們從 Reachy 的相機擷取單一影格並顯示出來！

In [ ]:
# 啟用多媒體連線（注意：此處不使用 media_backend="no_media"）
with ReachyMini() as mini:
    print("正在擷取影像...")

    # 從相機取得一個影格
    frame = mini.media.get_frame()
    # 相機初始化可能需要短暫時間，我們持續等待直到取得有效影格
    while frame is None:
        print("等待相機初始化...")
        time.sleep(0.5)
        frame = mini.media.get_frame()

    # 檢查是否成功取得有效影格
    if frame is not None:
        print("✓ 影像擷取成功！")
        print(f"  Resolution: {frame.shape[1]}x{frame.shape[0]}")
        print(f"  通道數: {frame.shape[2]} (BGR 格式)")
        print(f"  資料型態: {frame.dtype}")

        _, buffer = cv2.imencode(".jpg", frame)
        display(IPImage(data=buffer.tobytes()))
    else:
        print("✗ Failed to capture image")

**你剛剛做了什麼：**

* **`mini.media.get_frame()`**：返回包含影像資料的 NumPy 陣列
* **格式**：OpenCV 格式（BGR 顏色順序）
* **形狀 (Shape)**：`(高度, 寬度, 通道數)`
* **資料型態**：`uint8`（每個像素為 0-255）

**重要提示：** 影格為 **BGR 格式** (藍-綠-紅)。當使用 IPython.display 儲存或顯示時，OpenCV 會自動處理編碼轉換。

### 3. 將影像儲存至硬碟

讓我們將擷取到的影像儲存為檔案。

In [ ]:
with ReachyMini() as mini:
    frame = mini.media.get_frame()
    while frame is None:
        print("等待相機初始化...")
        time.sleep(0.5)
        frame = mini.media.get_frame()

    if frame is not None:
        # 使用 OpenCV 儲存（無需轉換格式 - 預設為 BGR）
        filename = f"reachy_photo_{int(time.time())}.jpg"
        cv2.imwrite(filename, frame)
        print(f"✓ 影像已儲存為: {filename}")
    else:
        print("✗ No frame to save")

你可以在目前的資料夾中找到它！

### 4. 即時相機串流畫面

你也可以使用 `clear_output` 直接在筆記本中串流相機畫面！

In [ ]:
from IPython.display import clear_output

STREAM_DURATION = 10  # seconds

with ReachyMini() as mini:
    # 等待相機初始化
    frame = mini.media.get_frame()
    while frame is None:
        time.sleep(0.1)
        frame = mini.media.get_frame()

    start_time = time.time()
    while time.time() - start_time < STREAM_DURATION:
        frame = mini.media.get_frame()
        if frame is not None:
            _, buffer = cv2.imencode(".jpg", frame)
            clear_output(wait=True)
            display(IPImage(data=buffer.tobytes()))
        time.sleep(0.033)  # ~30 fps

print("Stream ended!")

## 第 2 部分：音訊基礎 🎤🔊

### 5. 了解 Reachy 的音訊系統

Reachy Mini 配備：
* **麥克風陣列**：ReSpeaker 4 麥克風陣列
* **揚聲器**：用於播放聲音
* **取樣率 (Sample Rate)**：16 kHz（每秒 16,000 個取樣點）
* **格式**：依操作不同為單聲道 (Mono) 或雙聲道 (Stereo)

讓我們檢查音訊設定：

In [ ]:
with ReachyMini() as mini:
    input_rate = mini.media.get_input_audio_samplerate()
    output_rate = mini.media.get_output_audio_samplerate()

    print(f"輸入（麥克風）取樣率: {input_rate} Hz")
    print(f"Output (Speaker) Sample Rate: {output_rate} Hz")

### 6. 錄製音訊

讓我們從麥克風錄製 3 秒鐘的音訊。

**試試看：** 執行這個儲存格時說說話或拍拍手！

In [ ]:
RECORD_DURATION = 3  # seconds

with ReachyMini() as mini:
    print(f"正在錄音 {RECORD_DURATION} 秒...")
    print("🎤 請對著麥克風說話或發出聲音！")

    # 開始錄音
    mini.media.start_recording()

    audio_samples = []
    sample_rate = mini.media.get_input_audio_samplerate()
    target_samples = int(RECORD_DURATION * sample_rate)  # 我們需要的總取樣數
    total_samples_collected = 0

    # 持續收集音訊取樣直到達到目標數量
    while total_samples_collected < target_samples:
        sample = mini.media.get_audio_sample()

        if sample is not None:
            audio_samples.append(sample)
            total_samples_collected += len(sample)
            current_duration = total_samples_collected / sample_rate
            print(f"\r正在錄音... {current_duration:.1f}秒", end="")
        else:
            # 僅在等待新取樣時短暫睡眠
            time.sleep(0.01)

    # 停止錄音
    mini.media.stop_recording()

    print("\n✓ 錄音完成！")
    print(f"  共擷取 {len(audio_samples)} 個音訊區塊")

    # 合併並裁切至精確時長
    if audio_samples:
        audio_data = np.concatenate(audio_samples, axis=0)
        audio_data = audio_data[:target_samples]  # Trim to exact length
        print(f"  總音訊形狀 (Shape): {audio_data.shape}")
        print(f"  時長: {len(audio_data) / sample_rate:.2f} 秒")
    else:
        print("  ⚠ 未擷取到任何音訊資料")
        audio_data = None

**剛剛發生了什麼事：**

* **`start_recording()`**：啟用麥克風音訊擷取
* **`get_audio_sample()`**：返回一段音訊資料區塊（NumPy 陣列）
* **`stop_recording()`**：停用音訊擷取
* **資料格式**：float32 數值的 NumPy 陣列
* **取樣區塊**：每次呼叫 `get_audio_sample()` 都會返回一小段區塊（通常相當於 0.1-0.2 秒）

**重要概念：** 我們追蹤收集到的**音訊取樣總數**（而非實際經過的時間），以確保取得精確的錄音時長。接著我們會將最終音訊裁切為準確長度，避免錄進多餘的時間。

**效能提示：** 為了獲得最佳的反應速度，我們只在沒有可用取樣資料時才進行睡眠 (`sample is None`)。這讓我們能在音訊抵達時立即處理，這對於聲音偵測或監控等即時應用特別重要。

你也可以使用 IPython 的 `Audio` 元件直接在筆記本中播放剛錄好的音訊 — 不需要透過揚聲器輸出！

> **注意：** `Audio` 元件只有在**瀏覽器版的 Jupyter 伺服器**（終端機執行 `jupyter notebook` 或 `jupyter lab`）中才會發出聲音。在 VS Code 的筆記本編輯器中會顯示播放器但不會出聲。

In [ ]:
from IPython.display import Audio

if audio_data is not None:
    # Audio widget expects mono (1D) or channels-first (2, N) arrays.
    # The ReSpeaker returns (samples, channels), so we take the mean across channels.
    mono = audio_data.mean(axis=1) if audio_data.ndim > 1 else audio_data
    display(Audio(data=mono, rate=sample_rate))
else:
    print("No audio recorded yet. Run the recording cell first!")

### 7. 將音訊儲存為檔案

讓我們將錄製的音訊儲存為 WAV 檔案。

In [ ]:
if audio_data is not None and len(audio_data) > 0:
    filename = f"reachy_recording_{int(time.time())}.wav"
    sample_rate = 16000

    # Save with soundfile
    sf.write(filename, audio_data, sample_rate)
    print(f"✓ 音訊已儲存為: {filename}")
    print("  你可以使用任何播放器播放此檔案！")
else:
    print("No audio data to save. Run the recording cell first!")

### 8. 播放音訊

現在讓我們透過 Reachy 的揚聲器播放音訊！

為此，我們建立一個簡單的輔助函式：

In [ ]:
def play_audio_file(mini, audio_file_path):
    """Plays an audio file through Reachy's speaker.

    Args:
        mini: ReachyMini instance
        audio_file_path: Path to WAV file

    """
    # 載入音訊檔案
    data, _ = sf.read(audio_file_path, dtype="float32")

    # 開始播放
    mini.media.start_playing()
    print("🔊 正在播放音訊...")

    # 分區塊推送音訊樣本
    chunk_size = 1024
    for i in range(0, len(data), chunk_size):
        chunk = data[i : i + chunk_size]
        mini.media.push_audio_sample(chunk)

    # 等待播放完畢
    time.sleep(len(data) / mini.media.get_output_audio_samplerate())
    mini.media.stop_playing()
    print("✓ 播放完成！")


print("Helper function defined!")

它使用了 ReachyMini 的幾個不同函式：
- `get_output_audio_samplerate()`：獲取 ReachyMini 所需的取樣率，必要時可用於重取樣。
- `start_playing()`：啟動音訊播放
- `push_audio_sample()`：將音訊資料推送到輸出裝置
- `stop_playing()`：停止音訊播放


現在，讓我們用剛寫好的函式播放剛才錄製的音訊吧！

In [ ]:
# Make sure you've run the recording and saving cells first!
# Update this filename to match your saved recording
audio_file = "reachy_recording_1770979828.wav"  # Update with actual filename

# Or use the last saved file from the variables
if "filename" in dir():
    audio_file = filename
    print(f"使用檔案: {audio_file}")

try:
    with ReachyMini() as mini:
        play_audio_file(mini, audio_file)
except FileNotFoundError:
    print(f"⚠ 找不到檔案: {audio_file}")
    print("Make sure to run the recording and saving cells first!")

---
## 9. 產生並播放音調 (Tone)

讓我們透過程式生成一個簡單的嗶嗶聲 (Beep)！

In [ ]:
def generate_beep(frequency=440, duration=0.5, sample_rate=16000):
    """Generate a simple sine wave tone.

    Args:
        frequency: Frequency in Hz (440 = A note)
        duration: Duration in seconds
        sample_rate: Sample rate in Hz

    Returns:
        numpy array of audio samples

    """
    t = np.linspace(0, duration, int(sample_rate * duration))
    tone = 0.3 * np.sin(2 * np.pi * frequency * t)  # 0.3 = volume

    # Add fade in/out to avoid clicks
    fade_samples = int(sample_rate * 0.01)  # 10ms fade
    fade_in = np.linspace(0, 1, fade_samples)
    fade_out = np.linspace(1, 0, fade_samples)
    tone[:fade_samples] *= fade_in
    tone[-fade_samples:] *= fade_out

    return tone.astype(np.float32)


# Generate a beep
beep = generate_beep(frequency=880, duration=0.3)  # High A note

with ReachyMini() as mini:
    mini.media.start_playing()
    print("🔔 嗶！")

    # Push the beep
    chunk_size = 1024
    for i in range(0, len(beep), chunk_size):
        chunk = beep[i : i + chunk_size]
        mini.media.push_audio_sample(chunk)

    time.sleep(0.5)
    mini.media.stop_playing()
    print("Done!")

你可以嘗試修改參數來發出更高或更低的音調！

---

## 10. 調整音量

你可以透過 Daemon 的 **REST API** 調整 Reachy Mini 的揚聲器與麥克風音量。當聲音太小或太大時非常實用。

**注意：** 音量控制由 Daemon 管理，而非直接透過 Python SDK。我們使用 `requests` 函式庫來呼叫 API。

Daemon 的 URL 取決於你的硬體設置：
- **Lite**：`http://localhost:8000`
- **Wireless**：`http://reachy-mini.local:8000`（或機器人的 IP 位址）

In [ ]:
import requests

# Adjust this URL to match your setup:
# - Lite: "http://localhost:8000"
# - Wireless: "http://reachy-mini.local:8000" (or the robot's IP address)
DAEMON_URL = "http://reachy-mini.local:8000"

# Get current speaker volume
response = requests.get(f"{DAEMON_URL}/api/volume/current")
print(f"Speaker volume: {response.json()['volume']}%")

# Set speaker volume (0-100)
response = requests.post(f"{DAEMON_URL}/api/volume/set", json={"volume": 75})
print(f"Speaker volume set to: {response.json()['volume']}%")

# You can also control the microphone input volume:
response = requests.get(f"{DAEMON_URL}/api/volume/microphone/current")
print(f"Microphone volume: {response.json()['volume']}%")

**可用的音量端點：**

| 端點 | 方法 | 說明 |
|---|---|---|
| `/api/volume/current` | GET | 取得目前揚聲器音量 (0-100) |
| `/api/volume/set` | POST | 設定揚聲器音量 (`{"volume": 75}`) |
| `/api/volume/microphone/current` | GET | 取得目前麥克風音量 |
| `/api/volume/microphone/set` | POST | 設定麥克風音量 |

你也可以從 **Reachy Mini Control** 中調整音量。

**提示：** 你可以在 `http://{daemon-url}:8000/docs` 互動式瀏覽所有可用的 API 端點。

---

## 11. 結合多媒體與動作

讓我們把學到的東西結合起來，讓 Reachy 更有互動感！

**範例：帶有倒數計時的拍照環節**

這個範例建立了一個有趣的拍照流程，結合了：
* 🎵 **音訊**：倒數計時與成功時的嗶嗶聲
* 🤖 **動作**：頭部定位與天線擺動
* 📸 **相機**：擷取最終照片

機器人將會：
1. 移動到中立姿勢
2. 伴隨嗶聲與天線動作從 3 倒數到 1
3. 拍攝照片
4. 播放成功的提示音並做出慶祝動作
5. 顯示擷取到的照片

In [ ]:
# 「拍照」連貫動作：發出嗶聲、注視相機、拍照、再次嗶聲並慶祝
with ReachyMini() as mini:
    print("📸 拍照流程開始...")

    # 移動到中立姿勢
    mini.goto_target(head=create_head_pose(), antennas=[0.0, 0.0], duration=1.0)

    # 伴隨嗶聲與頭部天線動作進行倒數
    for i in [3, 2, 1]:
        print(f"  {i}...")

        # Beep
        beep = generate_beep(frequency=440 + i * 100, duration=0.2)
        mini.media.start_playing()
        for j in range(0, len(beep), 1024):
            mini.media.push_audio_sample(beep[j : j + 1024])

        # Wiggle antennas
        mini.goto_target(antennas=[0.3, -0.3], duration=0.3)
        mini.goto_target(antennas=[0.0, 0.0], duration=0.3)

        time.sleep(0.5)

    # 拍照！
    print("  📸 喀嚓！(拍照)")
    frame = mini.media.get_frame()

    # Success beep (higher pitch)
    success_beep = generate_beep(frequency=840, duration=0.4)
    mini.media.start_playing()
    for j in range(0, len(success_beep), 1024):
        mini.media.push_audio_sample(success_beep[j : j + 1024])
    time.sleep(0.5)
    # mini.media.stop_playing()

    # 慶祝動畫
    mini.goto_target(
        head=create_head_pose(pitch=-10, degrees=True),
        antennas=[0.5, -0.5],
        duration=0.5,
    )

    # 儲存並顯示
    if frame is not None:
        filename = f"reachy_selfie_{int(time.time())}.jpg"
        cv2.imwrite(filename, frame)
        print(f"\n✓ 照片已儲存: {filename}")

        # Display
        _, buffer = cv2.imencode(".jpg", frame)
        display(IPImage(data=buffer.tobytes()))

    # 回到中立姿勢
    mini.goto_target(head=create_head_pose(), antennas=[0.0, 0.0], duration=1.0)

---

## 12. 練習（自己動手試試看！）

### 練習 1：縮時攝影 (Time-Lapse Photography)

撰寫一個程式：
1. 每 5 秒拍一張照片
2. 總共拍攝 5 張照片
3. 每次拍照前播放嗶嗶聲
4. 使用流水號檔名儲存所有照片

**加分挑戰：** 在每張照片之間稍微移動頭部！

<details>
<summary><b>💡 點擊顯示解答</b></summary>

```python
beep = generate_beep()
chunk_size = 1024


with ReachyMini() as mini:
    frame = mini.media.get_frame()
    while frame is None:
        print("Waiting for camera to initialize...")
        time.sleep(0.5)
        frame = mini.media.get_frame()

    for i in range(5):
        starting_time = time.time()
        mini.media.start_playing()
        for j in range(0, len(beep), chunk_size):
            mini.media.push_audio_sample(beep[j:j+chunk_size])

        frame = mini.media.get_frame()

        filename = f"reachy_selfie_{int(time.time())}.jpg"
        cv2.imwrite(filename, frame)
        print(f"\n✓ Photo saved: {filename}")

        mini.goto_target(
            head=create_head_pose(roll=i*10-20),
            antennas=[i*0.1, -i*0.1],
            duration=0.5
        )

        while time.time() - starting_time < 5.0:
            time.sleep(0.1)
    
    mini.goto_target(
        head=create_head_pose(),
        antennas=[0.0, 0.0],
        duration=1.0
    )
``` 
</details>

In [ ]:
# 在此撰寫你的程式碼
with ReachyMini() as mini:
    # TODO: 實作縮時攝影功能
    pass

### 練習 2：音訊回音效果 (Audio Echo Effect)

錄製 2 秒鐘的音訊，然後連續播放兩次（做出類似回音的效果）。

**提示：** 你可以使用 `np.concatenate()` 來連接兩個音訊陣列。

<details>
<summary><b>💡 點擊顯示解答</b></summary>

```python
RECORD_DURATION = 2  # seconds

with ReachyMini() as mini:
    print(f"Recording for {RECORD_DURATION} seconds...")
    print("🎤 Say something!")

    # Start recording
    mini.media.start_recording()
    
    audio_samples = []
    sample_rate = mini.media.get_input_audio_samplerate()
    target_samples = int(RECORD_DURATION * sample_rate)  # Total samples we want
    total_samples_collected = 0
    
    # Collect audio samples until we have enough
    while total_samples_collected < target_samples:
        sample = mini.media.get_audio_sample()
        
        if sample is not None:
            audio_samples.append(sample)
            total_samples_collected += len(sample)
            current_duration = total_samples_collected / sample_rate
            print(f"\rRecording... {current_duration:.1f}s", end="")
            # Process samples as fast as possible - don't sleep when data is available
        else:
            # Only sleep when waiting for new samples
            time.sleep(0.01)
    
    mini.media.stop_recording()
    
    print(f"\n✓ Recording complete!")
    
    if audio_samples:
        audio_data = np.concatenate(audio_samples, axis=0)
        audio_data = audio_data[:target_samples]  # Trim to exact length
        
        # Convert stereo to mono if needed
        if audio_data.ndim > 1:
            audio_data = np.mean(audio_data, axis=1)
        
        # Create echo: original + silence gap + repeated audio
        silence = np.zeros(int(sample_rate * 0.3), dtype=np.float32)  # 0.3s gap
        echo = np.concatenate([audio_data, silence, audio_data])
        
        # Play back the echo
        mini.media.start_playing()
        print("🔊 Playing echo...")
        
        chunk_size = 1024
        for i in range(0, len(echo), chunk_size):
            mini.media.push_audio_sample(echo[i:i + chunk_size])
        
        time.sleep(len(echo) / sample_rate)
        mini.media.stop_playing()
        print("✓ Echo playback complete!")
    else:
        print("⚠ No audio data captured")
```

</details>

In [ ]:
# 在此撰寫你的程式碼
with ReachyMini() as mini:
    # TODO: 錄音並建立回音效果
    pass

### 練習 3：聲控自拍 (Sound-Triggered Selfie)

建立一個「安全監控相機」：
1. 持續監控環境音量
2. 當偵測到大聲響（高振幅）時自動拍照
3. 播放嗶聲表示拍照完成

**提示：** 使用 `np.abs(audio_sample).max()` 來偵測大聲響。

<details>
<summary><b>💡 點擊顯示解答</b></summary>

```python
AUDIO_THRESHOLD = 0.3  # Adjust based on your environment
LISTEN_DURATION = 15   # How long to listen (seconds)

beep = generate_beep(frequency=1000, duration=0.2)
chunk_size = 1024
photo_count = 0

with ReachyMini() as mini:
    print(f"🎤 Listening for loud sounds ({LISTEN_DURATION}s)...")
    print(f"   Threshold: {AUDIO_THRESHOLD}")
    print(f"   Note: ~100-200ms latency due to audio buffering")
    
    # Start recording to monitor audio
    mini.media.start_recording()
    
    sample_rate = mini.media.get_input_audio_samplerate()
    target_samples = int(LISTEN_DURATION * sample_rate)
    total_samples_collected = 0
    
    while total_samples_collected < target_samples:
        sample = mini.media.get_audio_sample()
        
        if sample is not None:
            total_samples_collected += len(sample)
            
            # Check audio level IMMEDIATELY for best responsiveness
            level = np.abs(sample).max()
            
            if level > AUDIO_THRESHOLD:
                current_time = total_samples_collected / sample_rate
                print(f"\n🔔 Sound detected at {current_time:.1f}s! (level: {level:.3f})")
                
                # Take a photo
                frame = mini.media.get_frame()
                if frame is not None:
                    photo_count += 1
                    filename = f"triggered_photo_{photo_count}.jpg"
                    cv2.imwrite(filename, frame)
                    print(f"📸 Photo saved: {filename}")
                
                # Play confirmation beep
                mini.media.start_playing()
                for i in range(0, len(beep), chunk_size):
                    mini.media.push_audio_sample(beep[i:i + chunk_size])
                time.sleep(0.3)
                mini.media.stop_playing()
                
                # Cooldown only after detection to avoid multiple triggers
                time.sleep(1.5)  # 1.5 second cooldown after a trigger
            # No sleep here - process next sample immediately for best responsiveness
        else:
            # Only sleep when no sample is available (avoids busy-waiting CPU)
            time.sleep(0.005)  # 5ms - very responsive
    
    mini.media.stop_recording()
    print(f"\n✓ Done! Captured {photo_count} photo(s).")
```

**效能說明：** 這段程式透過以下方式最佳化反應速度：
- 僅在沒有音訊取樣時睡眠
- 取樣一抵達便立即處理
- 使用極短的睡眠時間 (5ms)
- 因軟硬體音訊緩衝區限制，仍有約 100-200ms 的不可避免延遲

</details>

In [ ]:
# 在此撰寫你的程式碼
AUDIO_THRESHOLD = 0.3  # Adjust this value based on your environment

with ReachyMini() as mini:
    # TODO: 實作聲控自拍相機
    pass

### 練習 4：音樂問候 (Musical Greeting)

建立一個問候序列：
1. 播放一組 3-4 個音調（不同頻率）
2. 讓頭部與天線同步隨每個音調擺動
3. 在結尾拍一張照片

讓它充滿音樂節奏與趣味！

<details>
<summary><b>💡 點擊顯示解答</b></summary>

```python
# C major chord notes
notes = [
    (262, "C"),   # C4
    (330, "E"),   # E4
    (392, "G"),   # G4
    (523, "C"),   # C5
]

# Matching head/antenna poses for each note
poses = [
    (create_head_pose(yaw=-15, degrees=True), [-0.3, 0.3]),
    (create_head_pose(yaw=15, degrees=True),  [0.3, -0.3]),
    (create_head_pose(pitch=-10, degrees=True), [-0.5, 0.5]),
    (create_head_pose(roll=10, pitch=-10, degrees=True), [0.5, -0.5]),
]

chunk_size = 1024

with ReachyMini() as mini:
    # Start from neutral
    mini.goto_target(head=create_head_pose(), antennas=[0.0, 0.0], duration=1.0)
    
    print("🎵 Musical greeting!")
    for (freq, name), (head_pose, antennas) in zip(notes, poses):
        print(f"  ♪ {name} ({freq} Hz)")
        
        # Generate and play tone
        tone = generate_beep(frequency=freq, duration=0.4)
        mini.media.start_playing()
        for i in range(0, len(tone), chunk_size):
            mini.media.push_audio_sample(tone[i:i + chunk_size])
        
        # Move in sync with the note
        mini.goto_target(head=head_pose, antennas=antennas, duration=0.4)
        
        time.sleep(0.1)
        mini.media.stop_playing()
    
    # Take a photo at the end
    print("  📸 Say cheese!")
    frame = mini.media.get_frame()
    if frame is not None:
        filename = f"musical_greeting_{int(time.time())}.jpg"
        cv2.imwrite(filename, frame)
        print(f"  ✓ Photo saved: {filename}")
        
        _, buffer = cv2.imencode('.jpg', frame)
        display(IPImage(data=buffer.tobytes()))
    
    # Return to neutral
    mini.goto_target(head=create_head_pose(), antennas=[0.0, 0.0], duration=1.0)
    print("✓ Greeting complete!")
```

</details>

In [ ]:
# 在此撰寫你的程式碼
# Try frequencies like: 262 (C), 330 (E), 392 (G), 523 (C)

with ReachyMini() as mini:
    # TODO: 建立音樂問候動作
    pass